In [97]:
import  pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns 

# Decision Tree 
from sklearn.tree import DecisionTreeClassifier
#Random Forest 
from sklearn.ensemble import RandomForestClassifier
#Train Test
from sklearn.model_selection import train_test_split
# Data prepocessing 
from sklearn.preprocessing import StandardScaler, LabelEncoder
#Metrics 
from sklearn.metrics import classification_report, confusion_matrix

In [98]:
df = pd.read_csv('Dataset/student.csv')
df.head(2)

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,higher education,higher education,at_home,teacher,...,very good,medium,high,very low,very low,good,6,5,6,6
1,GP,F,17,U,GT3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,excellent,medium,medium,very low,very low,good,4,5,5,6


In [99]:
def convertMarks(marks):
    if marks >= 15 and marks <= 20:
        return 'A'
    elif marks >= 11 and marks < 15:
        return 'B'
    elif marks >= 7 and marks < 11:
        return 'C'
    else:
        return 'Fail'

In [100]:
df['Grade'] = df['G3'].apply(convertMarks)

In [101]:
le = LabelEncoder()
le_columns = ['schoolsup','famsup','internet']
def leConverter(dataframe,columns):
  for col in columns:
      dataframe[f'{col}_id'] = le.fit_transform(dataframe[col])
      print('Data converted successfully.')

In [102]:
leConverter(df, le_columns)

Data converted successfully.
Data converted successfully.
Data converted successfully.


In [103]:
medu={

'none': 0,   
'primary education(4th grade)':1,
'5th to 9th grade': 2,
'secondary education': 3,
'higher education':4
}
df['medu_id'] = df['Medu'].map(medu)

In [104]:
def manualEncoding(dataframe, col_name, *args):
    encoding_val ={}
    for idx, data in enumerate(args):
        encoding_val[data] = idx

    dataframe[f'{col_name}_id'] = dataframe[col_name].map(encoding_val)
    print(f'{col_name} data encoded.')

In [105]:
# Fedu 
manualEncoding(df, 'Fedu','none','primary education(4th grade)','5th to 9th grade','secondary education','higher education')
# Travel time
manualEncoding(df, 'traveltime','<15 min ','15 to 30min.','30min. to 1 hour','>1 hour')
# study Time 
manualEncoding(df, 'studytime','2 hours','2 to 5 hours','5 to 10 hours','>10 hours')
# free time and Go out
data = ['freetime','goout']
for col in data:
       manualEncoding(df,col,'very low','low','medium','high','very high')
manualEncoding(df,'health','very bad','bad','good','very good','excellent')
manualEncoding(df, 'Grade','Fail','C','B','A')

Fedu data encoded.
traveltime data encoded.
studytime data encoded.
freetime data encoded.
goout data encoded.
health data encoded.
Grade data encoded.


# Feature and Target

In [106]:
features = ['medu_id', 'Fedu_id', 'traveltime_id',
       'studytime_id', 'failures', 'schoolsup_id', 'famsup_id','internet_id','freetime_id','goout_id','health_id', 'absences','G1', 'G2']
target = 'Grade_id'

In [107]:
X = df[features]
Y = df[target]

# Train and test split 

In [108]:
X_train , X_test, Y_train , Y_test  = train_test_split(
X,Y, test_size = 0.2, random_state = 42, stratify=Y)

# Feature Scaling

In [109]:
scaler = StandardScaler()
X_train_scale = scaler.fit_transform(X_train)
X_test_scale = scaler.transform(X_test)

# Decision Tree Model

In [110]:
dt_model=DecisionTreeClassifier(max_depth=3,random_state=42)
dt_model.fit(X_train_scale,Y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,3
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [111]:
dt_pred = dt_model.predict(X_test_scale)

# Random Forest Model

In [112]:
rf_model = RandomForestClassifier(max_depth=5,n_estimators=100,class_weight='balanced',random_state=42)
rf_model.fit(X_train_scale, Y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,5
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [113]:
rf_pred = rf_model.predict(X_test_scale)

# metrics

## Decision Tree Metrics 

In [114]:
# Decision Tree Mterics 
dt_cr = classification_report(Y_test, dt_pred)

## Random Forest Mterics
rf_cr = classification_report(Y_test, rf_pred)

In [115]:
print(dt_cr)

              precision    recall  f1-score   support

           0       0.88      0.58      0.70        12
           1       0.75      0.96      0.84        25
           2       0.96      0.89      0.92        27
           3       1.00      0.93      0.97        15

    accuracy                           0.87        79
   macro avg       0.90      0.84      0.86        79
weighted avg       0.89      0.87      0.87        79



In [116]:
print(rf_cr)

              precision    recall  f1-score   support

           0       0.67      0.83      0.74        12
           1       0.83      0.80      0.82        25
           2       0.92      0.85      0.88        27
           3       0.93      0.93      0.93        15

    accuracy                           0.85        79
   macro avg       0.84      0.85      0.84        79
weighted avg       0.86      0.85      0.85        79



In [117]:
import pandas as pd

data = {
    'medu_id': [0, 2, 4],
    'Fedu_id': [2, 3, 1],
    'traveltime_id': [0, 1, 2],
    'studytime_id': [1, 1, 3],
    'failures': [2, 3, 1],
    'schoolsup_id': [0, 1, 1],
    'internet_id': [1, 1, 1],
    'freetime_id': [2, 1, 3],
    'famsup_id': [1, 1, 1],
    'goout_id': [1, 23, 2],  # fixed from 2 items to 3 items
    'health_id': [4, 3, 4],
    'absences': [3, 2, 0],
    'G1': [10, 12, 13],
    'G2': [15, 17, 6],
}

data_df = pd.DataFrame(data)
data_df

# features = [
#     'medu_id', 'Fedu_id', 'traveltime_id', 'studytime_id',
#     'failures', 'schoolsup_id', 'famsup_id', 'internet_id',
#     'freetime_id', 'goout_id', 'health_id', 'absences', 'G1', 'G2'
# ]

# X = data_df[features]
# print(X.shape)  # (3, 14)

,medu_id,Fedu_id,traveltime_id,studytime_id,failures,schoolsup_id,internet_id,freetime_id,famsup_id,goout_id,health_id,absences,G1,G2
0,0,2,0,1,2,0,1,2,1,1,4,3,10,15
1,2,3,1,1,3,1,1,1,1,23,3,2,12,17
2,4,1,2,3,1,1,1,3,1,2,4,0,13,6


In [120]:
data_scale = scaler.transform(data_df[scaler.feature_names_in_])
data_dt_pred = dt_model.predict(data_scale)
data_rf_pred = rf_model.predict(data_scale)

data_df['dt_pred'] = data_dt_pred
data_df['rf_pred'] = data_rf_pred

In [121]:
data_df

,medu_id,Fedu_id,traveltime_id,studytime_id,failures,schoolsup_id,internet_id,freetime_id,famsup_id,goout_id,health_id,absences,G1,G2,dt_pred,rf_pred
0,0,2,0,1,2,0,1,2,1,1,4,3,10,15,3,3
1,2,3,1,1,3,1,1,1,1,23,3,2,12,17,3,3
2,4,1,2,3,1,1,1,3,1,2,4,0,13,6,0,0
